# Step 7 — Evidence Synthesizer Agent
**Phase 3 | NIKKO Engineering Collective**

Spec: `SPEC-200 §5.5` | Reqs: `REQ-200-080`, `REQ-200-081`, `REQ-200-ER3`, `G-EVIDENCE-01`

This notebook exercises `agents/synthesizer_agent.py` against three canonical scenarios:

| Scenario | Setup | Expected |
|----------|-------|----------|
| **A** | PubMed (peer-reviewed) + WebSearch (grey-lit) mix | confidence ≥ 0.90, top citation is peer-reviewed |
| **B** | Grey-literature only (PubMed returned nothing) | confidence = 0.50, grey-lit hedging note in summary |
| **C** | Empty retrieval (Crisis Mode bypass) | confidence = 0.0, empty citations |


In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from docs.schemas.acp_schemas import EvidenceItem, EvidencePayload, EvidenceTier, SourceTier
from agents.synthesizer_agent import EvidenceSynthesizerAgent, _compute_confidence

agent = EvidenceSynthesizerAgent()
NOW = datetime.now(tz=timezone.utc)
print("Imports OK. Repo root:", repo_root)


In [ ]:
# Fixture helpers — build EvidenceItem / EvidencePayload without live API calls.
def make_item(title, abstract, source_name, evidence_tier, source_tier, pub_date=None):
    return EvidenceItem(
        title=title, abstract=abstract, url="https://example.com",
        source_name=source_name, publication_date=pub_date,
        evidence_tier=evidence_tier, source_tier=source_tier, retrieved_at=NOW,
    )

def make_payload(source_name, source_tier, items, grey_flag=False):
    return EvidencePayload(
        query="CBT for depression", source_name=source_name,
        source_tier=source_tier, results=items, grey_literature_flag=grey_flag,
    )

print("Helpers defined.")


---
## Scenario A — Peer-reviewed + grey-literature mix

PubMed returns two recent RCTs; WebSearch returns two sanctioned grey-lit pages.
One pair of items has similar titles (deduplication test case).

**Expected:** confidence ≥ 0.90 · top citation is peer-reviewed (G-EVIDENCE-01 tiebreak)


In [ ]:
pubmed_items = [
    make_item(
        "CBT for Major Depressive Disorder: A Systematic Review",
        "CBT demonstrates efficacy in reducing depressive symptoms across 42 randomised controlled "
        "trials. Effect size d equals 0.68 versus waitlist control. Remission rate 52 percent.",
        "PubMed Central", EvidenceTier.PEER_REVIEWED, SourceTier.PRIMARY, "2023-08-15",
    ),
    make_item(
        "Cognitive Behavioural Therapy for Depression and Anxiety: Meta-Analysis",
        "Combined CBT outperforms pharmacotherapy alone in randomised trials. Number needed "
        "to treat of four for treatment response. Sustained effects at 12 month follow-up.",
        "PubMed Central", EvidenceTier.PEER_REVIEWED, SourceTier.PRIMARY, "2021-03-01",
    ),
]

web_items = [
    make_item(
        "CBT for Depression and Anxiety: What the Evidence Shows",
        "Cognitive behavioural therapy is a first-line psychological treatment recommended "
        "by Australian mental health clinical practice guidelines for depression and anxiety.",
        "beyondblue.org.au", EvidenceTier.GREY_LITERATURE, SourceTier.PRIMARY, "2022-06-01",
    ),
    make_item(
        "Understanding Depression: Causes, Signs and Treatment Options",
        "Depression affects one in seven Australians during their lifetime. Effective evidence "
        "based treatments include cognitive behavioural therapy, medication, and lifestyle changes.",
        "healthdirect.gov.au", EvidenceTier.GREY_LITERATURE, SourceTier.PRIMARY, "2023-01-10",
    ),
]

result_A = agent.synthesize(
    [make_payload("PubMed Central", SourceTier.PRIMARY, pubmed_items),
     make_payload("Sanctioned Web Search", SourceTier.PRIMARY, web_items, grey_flag=True)],
    query="CBT for depression",
)

print(f"confidence:          {result_A.confidence}")
print(f"grey_literature_used: {result_A.grey_literature_used}")
print(f"source_disagreement:  {result_A.source_disagreement}")
print(f"citations count:      {len(result_A.citations)}")
print(f"top citation tier:    {result_A.citations[0].evidence_tier.value}")
print(f"top citation source:  {result_A.citations[0].source_name}")
print(f"\nSummary:\n{result_A.summary}")

assert result_A.confidence >= 0.90
assert result_A.grey_literature_used is True
assert result_A.citations[0].evidence_tier == EvidenceTier.PEER_REVIEWED
print("\n✓ Scenario A assertions passed.")


---
## Scenario B — Grey-literature only

PubMed returns zero results. WebSearch returns two sanctioned pages.

**Expected:** confidence = 0.50 (0.65 base − 0.15 dominant penalty) · grey-lit hedging note in summary


In [ ]:
grey_only_items = [
    make_item(
        "Depression: Signs, Symptoms and When to Seek Help",
        "Common signs of depression include persistent low mood, loss of interest in activities, "
        "fatigue, changes in sleep and appetite, and difficulty concentrating on daily tasks.",
        "healthdirect.gov.au", EvidenceTier.GREY_LITERATURE, SourceTier.PRIMARY, "2023-03-15",
    ),
    make_item(
        "Managing Depression with Psychological Therapies",
        "Cognitive behavioural therapy is recommended as first-line treatment for mild to moderate "
        "depression in Australian clinical practice guidelines. Sessions typically run 12 to 20 weeks.",
        "beyondblue.org.au", EvidenceTier.GREY_LITERATURE, SourceTier.PRIMARY, "2022-11-01",
    ),
]

result_B = agent.synthesize(
    [make_payload("PubMed Central", SourceTier.PRIMARY, []),           # empty PubMed
     make_payload("Sanctioned Web Search", SourceTier.PRIMARY, grey_only_items, grey_flag=True)],
    query="depression treatment",
)

print(f"confidence:           {result_B.confidence}  (expected 0.50)")
print(f"grey_literature_used: {result_B.grey_literature_used}")
print(f"citations count:      {len(result_B.citations)}")
print(f"\nSummary:\n{result_B.summary}")

# Confidence trace: 0.65 (grey-lit base) − 0.15 (no peer-reviewed) = 0.50
assert result_B.confidence == 0.50
assert "grey-literature" in result_B.summary.lower()
print("\n✓ Scenario B assertions passed.")


---
## Scenario C — Empty retrieval (Crisis Mode bypass)

Per `REQ-200-124`, evidence retrieval is deprioritised in Crisis Mode.
The Synthesizer is called with an empty list; it must still return a valid
zero-confidence bundle so the pipeline can assemble a safe crisis response.


In [ ]:
result_C = agent.synthesize([], query="")

print(f"confidence:   {result_C.confidence}  (expected 0.0)")
print(f"citations:    {result_C.citations}")
print(f"summary:      {result_C.summary}")

assert result_C.confidence == 0.0
assert result_C.citations == []
print("\n✓ Scenario C assertions passed.")


---
## Confidence formula unit checks

Verify every scoring rule in isolation before it runs inside the full pipeline.


In [ ]:
peer = make_item("T", "A", "PubMed", EvidenceTier.PEER_REVIEWED, SourceTier.PRIMARY)
grey = make_item("T", "A", "healthdirect", EvidenceTier.GREY_LITERATURE, SourceTier.PRIMARY)

cases = [
    # description, citations, grey_lit_used, disagree, expected
    ("Peer-reviewed, no penalties",           [peer],        False, False, 0.90),
    ("Peer-reviewed + disagreement penalty",  [peer],        False, True,  0.80),
    ("Grey-lit only, no disagreement",        [grey],        True,  False, 0.50),  # 0.65−0.15
    ("Grey-lit only + disagreement",          [grey],        True,  True,  0.40),  # 0.65−0.15−0.10
    ("Mixed peer+grey, no disagreement",      [peer, grey],  True,  False, 0.90),
    ("Empty citations",                       [],            False, False, 0.00),
]

print(f"{'Description':<45} {'Expected':>9} {'Got':>9} {'Pass':>6}")
print("─" * 75)
all_pass = True
for desc, cits, gl, dis, exp in cases:
    got = _compute_confidence(cits, gl, dis)
    ok = abs(got - exp) < 1e-9
    all_pass = all_pass and ok
    print(f"{desc:<45} {exp:>9.2f} {got:>9.4f} {'✓' if ok else '✗':>6}")

assert all_pass
print("\n✓ All confidence formula checks passed.")


---
## Step 7 complete

| Check | REQ | Status |
|-------|-----|--------|
| Deterministic synthesis — no LLM call | design choice | ✅ |
| Scenario A: peer + grey mix, peer ranks first | G-EVIDENCE-01 | ✅ |
| Scenario B: grey-only confidence penalty applied | REQ-200-ER3 | ✅ |
| Scenario C: empty/Crisis Mode bypass returns valid bundle | REQ-200-124 | ✅ |
| Confidence formula — 6 unit cases | REQ-200-080 | ✅ |
| Deduplication by Jaccard title-token overlap | REQ-200-080 | ✅ |
| Disagreement detection with min-token guard | REQ-200-072 | ✅ |
| No emotion interpretation, no advice, no routing | REQ-200-081 | ✅ |

**Next:** Step 8 — Evaluator Agent (per-response content gate).
